# 06 · Stable Audio 3: 텍스트 → 음악 (CFG 비교)

**Stable Audio 3 Medium** (Stability AI, 2026). 텍스트 인코더(T5Gemma)가 프롬프트를 숫자로 바꾸고, 확산 트랜스포머(DiT)가 노이즈를 8단계에 걸쳐 음악의 압축 표현(latent)으로 다듬은 뒤, 오디오 디코더(SAME-L)가 44.1 kHz 스테레오로 되돌립니다. 이 노트북은 공식 CPU 런타임(LiteRT)을 써서 **GPU와 로그인 없이** 돌아갑니다.

- 입력 → 출력: 텍스트 프롬프트 → 44.1 kHz 스테레오 음악
- 소요 시간: 설치 1분 + 가중치 내려받기 2~3분(처음 한 번) + 20초 곡 1개 CPU 2코어 기준 약 45초 (CFG가 1이 아니면 약 65초)
- GPU 불필요. 가중치 라이선스는 Stability AI Community License (상업 서비스에 쓸 때 확인)
- 논문: Evans et al., "Stable Audio 3," 2026 (arXiv:2605.17991) · 코드: https://github.com/Stability-AI/stable-audio-3

In [ ]:
#@title 0. 준비: 수업 자료 받기와 모델 폴더 정하기 (실행만 하면 됩니다)
#@markdown 모델 가중치는 모두 `models` 폴더 하나에 받습니다. 기본 위치는 `/content`이고 런타임을 끄면 지워집니다. 체크하면 Google Drive의 `CT-AI_Microdegree/` 아래에 `models`와 `outputs`를 두어 다음 실행 때 다시 받지 않고, 다른 노트북의 결과도 이어서 쓸 수 있습니다.
KEEP_IN_DRIVE = False  #@param {type:"boolean"}
import os, sys, glob, time, json, subprocess
_cands = [os.environ.get('CTAI_REPO_DIR'), os.path.abspath(os.path.join(os.getcwd(), '..', '..')), '/content/CT-AI_Microdegree']
REPO = next((c for c in _cands if c and os.path.isfile(os.path.join(c, 'colab', 'ctai.py'))), None)
if REPO is None:
    REPO = '/content/CT-AI_Microdegree'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/dabinkim0/CT-AI_Microdegree', REPO], check=True)
sys.path.insert(0, os.path.join(REPO, 'colab'))
REQ = os.path.join(REPO, 'colab', 'requirements')   # 노트북별 pip 패키지 목록
import ctai
P = ctai.setup(keep_in_drive=KEEP_IN_DRIVE)
AUDIO_DIR, ASSET_DIR, MODELS_DIR, CODE_DIR, OUT_DIR, IN_COLAB = P['AUDIO_DIR'], P['ASSET_DIR'], P['MODELS_DIR'], P['CODE_DIR'], P['OUT_DIR'], P['IN_COLAB']
print('예시 음원:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(AUDIO_DIR, '*.wav'))))


In [ ]:
#@title 설치와 모델 준비 (약 1분, 가중치는 처음 실행할 때 내려받음)
#@markdown `medium`(1.4 B, 내려받기 약 3.2 GB + 첫 실행 때 만드는 CPU 캐시 약 2.3 GB)이 기본입니다. 느리면 `sm-music`(작은 음악 모델, 약 1.2 GB, 최대 120초)을 고르세요.
MODEL = 'medium'  #@param ["medium", "sm-music"]
%pip install -q -r {REQ}/sa3.txt
import sa3
sa3.prepare(MODELS_DIR, OUT_DIR, dit=MODEL, decoder='same-l' if MODEL == 'medium' else 'same-s')
print('설치 OK')

## 비교 설정

같은 **프롬프트·시드·길이·스텝**에서 CFG만 바꿔 생성합니다.

- `CFG`는 프롬프트를 얼마나 세게 따를지 정합니다. 1은 기본 생성, 1보다 크면 프롬프트 쪽으로 더 끌어당깁니다. 값이 크다고 품질이 좋아지지는 않으니 음량과 왜곡도 같이 들어 보세요.
- 같은 시드라도 CFG가 다르면 편곡이나 멜로디가 달라질 수 있습니다.
- `NEGATIVE_PROMPT`는 CFG가 1일 때는 쓰이지 않습니다. 쓰려면 목록에서 1을 빼세요.
- 프롬프트에는 장르, 악기, 분위기, BPM을 넣습니다. 이 모델은 알아들을 수 있는 가사는 만들지 못합니다.

In [ ]:
#@title 프롬프트와 설정
PROMPT = "Instrumental cinematic ambient background music, warm felt piano arpeggios, soft evolving analog pads, subtle strings, slow 75 BPM, peaceful and hopeful, gentle dynamic development, spacious clean mix, no vocals, no speech."  #@param {type:"string"}
SECONDS = 20  #@param {type:"integer"}
SEED = 42     #@param {type:"integer"}
STEPS = 8     #@param {type:"integer"}
CFG_VALUES = [1.0, 4.0]   # 시간이 있으면 [1.0, 2.0, 4.0, 6.0]
NEGATIVE_PROMPT = ""

In [ ]:
#@title 생성: CFG 값마다 한 곡씩 차례로
t0 = time.time()
comparison_dir, cfg_clips = sa3.generate_cfg_sweep(PROMPT, cfg_values=CFG_VALUES, seconds=SECONDS, seed=SEED,
                                                   steps=STEPS, negative_prompt=NEGATIVE_PROMPT)
print('\n총 %.0f s · 결과 폴더: %s' % (time.time() - t0, comparison_dir))

## 같은 축·같은 색 범위의 멜 스펙트로그램 + 오디오

모든 결과를 같은 시간·주파수 축과 같은 dB 범위(-90 ~ -10 dB)로 그리고, 음량을 따로 맞추지 않습니다. 그래서 그림이 밝은 것은 음색 차이일 수도 있고 **음량이 커진 것**일 수도 있습니다. 플레이어 위의 RMS·Peak 숫자를 같이 보세요.

In [ ]:
#@title 비교: 스펙트로그램과 플레이어 (PNG·HTML·JSON 저장)
cfg_report = sa3.compare_audio(cfg_clips, title=f"CFG comparison | seed {SEED}", output_dir=comparison_dir, filename="cfg_comparison")

## 해 보기

1. **같은 구간만 확대**: 아래 셀의 `CROP_RANGE`로 모든 결과의 같은 구간을 나란히 듣습니다. CFG가 커질수록 어떤 소리가 커지거나 거칠어지나요?
2. **프롬프트 한 구절만 바꾸기**: `75 BPM`을 `120 BPM`으로, `felt piano`를 `nylon guitar`로 바꾸고 다시 생성해 보세요. 모델이 숫자(BPM)를 실제로 따르나요?
3. **시드 바꾸기**: `SEED`만 바꾸면 같은 프롬프트로 다른 곡이 나옵니다. 같은 시드와 같은 설정이면 같은 결과가 나옵니다.

In [ ]:
#@title 같은 구간 확대 비교
CROP_RANGE = (5.0, 15.0)
sa3.compare_audio(cfg_clips, crop=CROP_RANGE, title="CFG comparison | selected interval", output_dir=comparison_dir, filename="cfg_comparison_crop")

In [ ]:
#@title 저장: 결과 폴더를 zip으로 내려받기
import shutil
zip_path = shutil.make_archive(str(comparison_dir), 'zip', str(comparison_dir))
print('저장:', zip_path, '(cfg_comparison.html을 브라우저로 열면 그림과 소리를 같이 볼 수 있습니다)')
if IN_COLAB and not OUT_DIR.startswith('/content/drive'):
    from google.colab import files; files.download(zip_path)